## Modelling 

In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import TrainingArguments, Trainer
from torch.utils.data import Dataset

### 1. DATA LOADING & PREPROCESSING

In [15]:

print("Loading data...")
df_full = pd.read_csv('../datasets/processed/movies_dataset_ready.csv')

# Karena keterbatasan waktu maka ambil 5000 data dulu buat eksperimen 
df = df_full.sample(n=5000, random_state=42).reset_index(drop=True)
# Membuat list genre dan Encode pake MultiLabelBinarizer
df['genre_list'] = df['genres'].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
mlb = MultiLabelBinarizer()
labels = mlb.fit_transform(df['genre_list'])
num_labels = len(mlb.classes_)

# Split Data 80 training : 20 testing 
X = df['content'].to_numpy()
y = labels
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Total data sekarang: {len(df)} baris.")

Loading data...
Total data sekarang: 5000 baris.


### 2.Tokenizer and Dataset Class

In [16]:
model_name = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizer.from_pretrained(model_name)

class MovieDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256): 
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.FloatTensor(label) 
        }

train_dataset = MovieDataset(X_train, y_train, tokenizer)
test_dataset = MovieDataset(X_test, y_test, tokenizer)

### 3.Setup Metriks

In [17]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    # Karena multi-label, maka memakai Sigmoid buat mendapatkan probabilitas (0-1)
    probs = 1 / (1 + np.exp(-logits))
    
    # Threshold 0.5 buat menentukan dia masuk ke genre itu atau tidak
    predictions = np.where(probs > 0.5, 1, 0)
    
    # Hitung metrik (micro dan macro average)
    f1_micro = f1_score(labels, predictions, average='micro', zero_division=0)
    precision = precision_score(labels, predictions, average='micro', zero_division=0)
    recall = recall_score(labels, predictions, average='micro', zero_division=0)
    
    return {
        'f1_micro': f1_micro,
        'precision': precision,
        'recall': recall
    }

### 4. Setup Model dan Training Arguments

In [18]:
is_cuda_ready = torch.cuda.is_available()
print(f"is CUDA ready? {is_cuda_ready}")

is CUDA ready? True


In [19]:
print("Setting up Model & Trainer...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = DistilBertForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=num_labels, 
    problem_type="multi_label_classification"
)
model.to(device)

# BEST PARAMETER
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=16,
    num_train_epochs=3, 
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    logging_dir='./logs',
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

Setting up Model & Trainer...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


### 5. Training and Evaluation

In [20]:
print("Mulai Training...")
trainer.train()

print("\n--- HASIL METRIK TERBAIK ---")
eval_results = trainer.evaluate()
print(eval_results)

Mulai Training...


Epoch,Training Loss,Validation Loss,F1 Micro,Precision,Recall
1,0.207007,0.183136,0.650355,0.995169,0.483001
2,0.119367,0.104716,0.819061,0.996639,0.695193
3,0.098204,0.087017,0.872691,0.997738,0.775498


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- HASIL METRIK TERBAIK ---


Training Loss,Validation Loss,Epoch,F1 Micro,Precision,Recall
0.098204,0.087017,3,0.872691,0.997738,0.775498


{'eval_loss': 0.08701661229133606, 'eval_f1_micro': 0.8726912928759895, 'eval_precision': 0.997737556561086, 'eval_recall': 0.7754982415005862}


### 6. Menyimpan model dan parameter terbaik

In [21]:
save_path = '../models/best_model_v1'
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"\nModel dan Parameter terbaik sudah disimpen di folder: {save_path} 📁")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model dan Parameter terbaik sudah disimpen di folder: ../models/best_model_v1 📁
